In [3]:
%pip install numpy pandas openpyxl plotly


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# ============================================================
# AFM HTML DASHBOARD GENERATOR
# ============================================================
#
# This script:
#
# 1. Searches every AFM subfolder
# 2. Reads every AFM text curve
# 3. Separates approach and retract
# 4. Converts position to nm
# 5. Converts force to nN
# 6. Reads kB, E and RMS from the matching Excel sheet
# 7. Creates one interactive HTML dashboard
#
# Output:
#
# AFM_dashboard.html
#
# ============================================================


from pathlib import Path
import re
import json

import numpy as np
import pandas as pd

from plotly.offline import get_plotlyjs


# ============================================================
# FOLDER LOCATIONS
# ============================================================

AFM_FOLDER = Path("afm_data")

EXCEL_FOLDER = Path("afm_values")

OUTPUT_HTML = Path("AFM_dashboard.html")


# ============================================================
# DEFAULT GRAPH RANGE
# ============================================================

DEFAULT_X_MIN = -150

DEFAULT_X_MAX = 200


# ============================================================
# READ ONE AFM TEXT FILE
# ============================================================

def load_afm_txt(file_path):

    segments = {}

    current_segment = None

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:

        for line in file:

            line = line.strip()

            # Detect AFM segment
            if line.startswith("# segment:"):

                current_segment = (
                    line
                    .split(":", 1)[1]
                    .strip()
                    .lower()
                )

                segments.setdefault(
                    current_segment,
                    []
                )

                continue

            # Ignore blank lines
            if not line:
                continue

            # Ignore header lines
            if line.startswith("#"):
                continue

            if current_segment is None:
                continue

            try:

                row = [
                    float(value)
                    for value in line.split()
                ]

            except ValueError:

                continue

            if len(row) >= 2:

                segments[
                    current_segment
                ].append(row)


    approach = np.asarray(
        segments.get("extend", []),
        dtype=float
    )

    retract = np.asarray(
        segments.get("retract", []),
        dtype=float
    )

    return approach, retract


# ============================================================
# FIND CURVE NUMBER FROM FILE NAME
# ============================================================
#
# Example:
#
# qi-data-xxxx-cropped_157.txt
#
# becomes:
#
# 157
#
# ============================================================

def extract_curve_number(file_path):

    match = re.search(
        r"cropped_(\d+)\.txt$",
        file_path.name,
        re.IGNORECASE
    )

    if match:

        return int(
            match.group(1)
        )

    # Backup method
    match = re.search(
        r"_(\d+)\.txt$",
        file_path.name,
        re.IGNORECASE
    )

    if match:

        return int(
            match.group(1)
        )

    return None


# ============================================================
# WORK OUT EXCEL FILE AND SHEET
# ============================================================
#
# Example:
#
# +ve_cell1part2
#
# becomes:
#
# Excel file = +ve.xlsx
#
# Sheet = cell1part2
#
# ============================================================

def get_excel_mapping(file_path):

    folder_name = (
        file_path.parent.name
    )

    if "_" not in folder_name:

        return None, None

    condition, sheet_name = (
        folder_name.split(
            "_",
            1
        )
    )

    excel_file = (
        EXCEL_FOLDER
        /
        f"{condition}.xlsx"
    )

    return excel_file, sheet_name


# ============================================================
# EXCEL CACHE
# ============================================================
#
# Excel sheets are stored after the first read.
#
# This speeds up processing.
#
# ============================================================

excel_cache = {}


# ============================================================
# READ VALUES FOR ONE CURVE
# ============================================================

def get_curve_values(file_path):

    curve_number = (
        extract_curve_number(
            file_path
        )
    )

    result = {

        "curve": curve_number,

        "kB": None,

        "E": None,

        "RMS": None
    }

    if curve_number is None:

        return result


    excel_file, sheet_name = (
        get_excel_mapping(
            file_path
        )
    )

    if excel_file is None:

        return result

    if not excel_file.exists():

        return result


    cache_key = (
        str(excel_file),
        sheet_name
    )


    if cache_key not in excel_cache:

        try:

            df = pd.read_excel(
                excel_file,
                sheet_name=sheet_name
            )

            # Keep only useful columns
            required_columns = [

                "Curve",

                "kB (nN/µm)",

                "E (MPa)",

                "RMS (pN)"
            ]

            available_columns = [

                column

                for column
                in required_columns

                if column in df.columns
            ]

            df = df[
                available_columns
            ].copy()

            excel_cache[
                cache_key
            ] = df

        except Exception as error:

            print(
                "Could not read:",
                excel_file,
                sheet_name,
                error
            )

            return result


    df = excel_cache[
        cache_key
    ].copy()


    if "Curve" not in df.columns:

        return result


    df["Curve"] = pd.to_numeric(
        df["Curve"],
        errors="coerce"
    )


    row = df[
        df["Curve"] == curve_number
    ]


    if row.empty:

        return result


    row = row.iloc[0]


    def safe_number(value):

        try:

            number = float(value)

            if np.isfinite(number):

                return number

        except Exception:

            pass

        return None


    result["kB"] = safe_number(
        row.get(
            "kB (nN/µm)"
        )
    )

    result["E"] = safe_number(
        row.get(
            "E (MPa)"
        )
    )

    result["RMS"] = safe_number(
        row.get(
            "RMS (pN)"
        )
    )


    return result


# ============================================================
# CONVERT ONE AFM FILE INTO DASHBOARD DATA
# ============================================================

def process_curve(file_path):

    approach, retract = (
        load_afm_txt(
            file_path
        )
    )

    values = (
        get_curve_values(
            file_path
        )
    )


    # --------------------------------------------------------
    # Approach data
    # --------------------------------------------------------

    approach_x = []

    approach_y = []


    if approach.size > 0:

        approach_x = (
            approach[:, 0]
            *
            1e9
        ).tolist()

        approach_y = (
            approach[:, 1]
            *
            1e9
        ).tolist()


    # --------------------------------------------------------
    # Retract data
    # --------------------------------------------------------

    retract_x = []

    retract_y = []


    if retract.size > 0:

        retract_x = (
            retract[:, 0]
            *
            1e9
        ).tolist()

        retract_y = (
            retract[:, 1]
            *
            1e9
        ).tolist()


    curve_number = (
        values["curve"]
    )


    if curve_number is None:

        curve_label = (
            file_path.stem
        )

    else:

        curve_label = (
            f"_{curve_number:03d}"
        )


    return {

        "filename":
            file_path.name,

        "curve_number":
            curve_number,

        "curve_label":
            curve_label,

        "kB":
            values["kB"],

        "E":
            values["E"],

        "RMS":
            values["RMS"],

        "approach_x":
            approach_x,

        "approach_y":
            approach_y,

        "retract_x":
            retract_x,

        "retract_y":
            retract_y
    }


# ============================================================
# FIND ALL AFM FOLDERS
# ============================================================

folder_paths = sorted({

    file.parent

    for file
    in AFM_FOLDER.rglob("*.txt")

})


if not folder_paths:

    raise FileNotFoundError(

        f"No AFM text files found in "
        f"{AFM_FOLDER.resolve()}"
    )


# ============================================================
# PROCESS COMPLETE DATASET
# ============================================================

dashboard_data = {}


for folder in folder_paths:

    folder_name = str(
        folder.relative_to(
            AFM_FOLDER
        )
    )


    files = list(
        folder.glob("*.txt")
    )


    # --------------------------------------------------------
    # Sort by numerical curve number
    #
    # Highest curve number appears first
    # --------------------------------------------------------

    files = sorted(

        files,

        key=lambda file: (

            extract_curve_number(file)

            if extract_curve_number(file)
            is not None

            else -1
        ),

        reverse=True
    )


    print(
        "Processing:",
        folder_name,
        f"({len(files)} curves)"
    )


    curves = []


    for file_path in files:

        curves.append(

            process_curve(
                file_path
            )
        )


    dashboard_data[
        folder_name
    ] = curves


# ============================================================
# CONVERT PYTHON DATA TO JSON
# ============================================================

data_json = json.dumps(
    dashboard_data,
    allow_nan=False
)


# ============================================================
# INCLUDE PLOTLY INSIDE HTML
# ============================================================
#
# The resulting HTML contains Plotly itself.
#
# A separate Plotly installation is not needed when someone
# opens the finished HTML page.
#
# ============================================================

plotly_js = get_plotlyjs()


# ============================================================
# BUILD HTML PAGE
# ============================================================

html = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>AFM Dashboard</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 0;
    background: #f5f5f5;
}}

.header {{
    background: white;
    padding: 18px 28px;
    border-bottom: 1px solid #ddd;
}}

.header h1 {{
    margin: 0;
    font-size: 24px;
}}

.controls {{
    background: white;
    margin: 20px;
    padding: 18px;
    border-radius: 8px;
}}

.control-row {{
    display: flex;
    gap: 14px;
    align-items: center;
    flex-wrap: wrap;
    margin-bottom: 12px;
}}

label {{
    font-weight: bold;
}}

select,
input,
button {{
    padding: 8px 10px;
    font-size: 14px;
}}

select {{
    min-width: 300px;
}}

button {{
    cursor: pointer;
}}

.main {{
    display: grid;
    grid-template-columns: minmax(600px, 1fr) 260px;
    gap: 20px;
    margin: 20px;
}}

.plot-card {{
    background: white;
    border-radius: 8px;
    padding: 15px;
}}

.value-card {{
    background: white;
    border-radius: 8px;
    padding: 22px;
    height: fit-content;
}}

.value-card h2 {{
    margin-top: 0;
    font-size: 18px;
}}

.value {{
    margin-bottom: 24px;
}}

.value-name {{
    font-size: 14px;
    color: #555;
}}

.value-number {{
    font-size: 22px;
    margin-top: 4px;
}}

.filename {{
    font-size: 12px;
    color: #666;
    overflow-wrap: anywhere;
}}

.counter {{
    font-size: 13px;
    color: #555;
}}

@media (max-width: 900px) {{

    .main {{
        grid-template-columns: 1fr;
    }}

}}

</style>


<script>

{plotly_js}

</script>

</head>


<body>


<div class="header">

    <h1>AFM Force Indentation Dashboard</h1>

</div>


<div class="controls">


    <div class="control-row">

        <label>Folder</label>

        <select id="folderSelect"></select>

    </div>


    <div class="control-row">

        <label>Curve</label>

        <select id="curveSelect"></select>

    </div>


    <div class="control-row">

        <button id="previousButton">
            Previous
        </button>

        <button id="nextButton">
            Next
        </button>

        <span
            class="counter"
            id="curveCounter">
        </span>

    </div>


    <div class="control-row">

        <label>X min (nm)</label>

        <input
            id="xMin"
            type="number"
            value="{DEFAULT_X_MIN}"
        >

        <label>X max (nm)</label>

        <input
            id="xMax"
            type="number"
            value="{DEFAULT_X_MAX}"
        >

        <button id="applyAxis">
            Apply range
        </button>

    </div>


</div>


<div class="main">


    <div class="plot-card">

        <div
            id="afmPlot"
            style="width:100%; height:620px;">
        </div>

    </div>


    <div class="value-card">

        <h2>Curve values</h2>


        <div class="value">

            <div class="value-name">
                kB
            </div>

            <div
                class="value-number"
                id="kbValue">
            </div>

            <div>
                nN/µm
            </div>

        </div>


        <div class="value">

            <div class="value-name">
                E
            </div>

            <div
                class="value-number"
                id="eValue">
            </div>

            <div>
                MPa
            </div>

        </div>


        <div class="value">

            <div class="value-name">
                RMS
            </div>

            <div
                class="value-number"
                id="rmsValue">
            </div>

            <div>
                pN
            </div>

        </div>


        <div class="value">

            <div class="value-name">
                Curve
            </div>

            <div
                class="value-number"
                id="curveValue">
            </div>

        </div>


        <div
            class="filename"
            id="filenameValue">
        </div>


    </div>


</div>


<script>


const dataset = {data_json}


const folderSelect =
    document.getElementById("folderSelect")


const curveSelect =
    document.getElementById("curveSelect")


const kbValue =
    document.getElementById("kbValue")


const eValue =
    document.getElementById("eValue")


const rmsValue =
    document.getElementById("rmsValue")


const curveValue =
    document.getElementById("curveValue")


const filenameValue =
    document.getElementById("filenameValue")


const curveCounter =
    document.getElementById("curveCounter")


const xMin =
    document.getElementById("xMin")


const xMax =
    document.getElementById("xMax")



function displayNumber(value, digits) {{

    if (
        value === null ||
        value === undefined
    ) {{

        return "N/A"
    }}

    return Number(value).toFixed(digits)
}}



function populateFolders() {{

    const folders =
        Object.keys(dataset)


    folders.forEach(folder => {{

        const option =
            document.createElement("option")

        option.value =
            folder

        option.textContent =
            folder

        folderSelect.appendChild(
            option
        )

    }})

}}



function populateCurves() {{

    const folder =
        folderSelect.value


    const curves =
        dataset[folder]


    curveSelect.innerHTML =
        ""


    curves.forEach(
        (curve, index) => {{

            const option =
                document.createElement("option")

            option.value =
                index

            option.textContent =
                curve.curve_label

            curveSelect.appendChild(
                option
            )

        }}
    )


    curveSelect.value =
        0


    updatePlot()
}}



function updatePlot() {{

    const folder =
        folderSelect.value


    const curves =
        dataset[folder]


    const index =
        Number(
            curveSelect.value
        )


    const curve =
        curves[index]


    if (!curve) {{

        return
    }}


    const approachTrace = {{

        x:
            curve.approach_x,

        y:
            curve.approach_y,

        type:
            "scatter",

        mode:
            "lines",

        name:
            "Approach",

        line: {{
            color: "black",
            width: 1.5
        }}
    }}


    const retractTrace = {{

        x:
            curve.retract_x,

        y:
            curve.retract_y,

        type:
            "scatter",

        mode:
            "lines",

        name:
            "Retract",

        line: {{
            color: "red",
            width: 1.5
        }}
    }}


    const layout = {{

        title: {{
            text:
                curve.filename
        }},

        xaxis: {{

            title:
                "Indentation (nm)",

            range: [

                Number(
                    xMin.value
                ),

                Number(
                    xMax.value
                )
            ],

            zeroline: true
        }},

        yaxis: {{

            title:
                "Force (nN)",

            zeroline: true
        }},

        margin: {{
            l: 80,
            r: 30,
            t: 70,
            b: 70
        }},

        legend: {{
            x: 0.02,
            y: 0.98
        }},

        hovermode:
            "closest"
    }}


    const config = {{

        responsive: true,

        displaylogo: false,

        toImageButtonOptions: {{

            format:
                "png",

            filename:
                curve.filename
                    .replace(
                        ".txt",
                        ""
                    ),

            scale:
                2
        }}
    }}


    Plotly.react(

        "afmPlot",

        [
            approachTrace,
            retractTrace
        ],

        layout,

        config
    )


    kbValue.textContent =
        displayNumber(
            curve.kB,
            3
        )


    eValue.textContent =
        displayNumber(
            curve.E,
            3
        )


    rmsValue.textContent =
        displayNumber(
            curve.RMS,
            2
        )


    curveValue.textContent =
        curve.curve_label


    filenameValue.textContent =
        curve.filename


    curveCounter.textContent =

        "Curve "
        +
        (index + 1)
        +
        " of "
        +
        curves.length

}}



function previousCurve() {{

    let index =
        Number(
            curveSelect.value
        )


    if (index > 0) {{

        index -= 1

        curveSelect.value =
            index

        updatePlot()
    }}

}}



function nextCurve() {{

    const folder =
        folderSelect.value


    const curves =
        dataset[folder]


    let index =
        Number(
            curveSelect.value
        )


    if (
        index <
        curves.length - 1
    ) {{

        index += 1

        curveSelect.value =
            index

        updatePlot()
    }}

}}



folderSelect.addEventListener(

    "change",

    populateCurves
)


curveSelect.addEventListener(

    "change",

    updatePlot
)


document
    .getElementById(
        "previousButton"
    )
    .addEventListener(

        "click",

        previousCurve
    )


document
    .getElementById(
        "nextButton"
    )
    .addEventListener(

        "click",

        nextCurve
    )


document
    .getElementById(
        "applyAxis"
    )
    .addEventListener(

        "click",

        updatePlot
    )



populateFolders()

populateCurves()


</script>


</body>

</html>
"""


# ============================================================
# SAVE HTML
# ============================================================

OUTPUT_HTML.write_text(
    html,
    encoding="utf-8"
)


print()

print(
    "Dashboard created successfully"
)

print(
    "File:",
    OUTPUT_HTML.resolve()
)

print(
    "Folders:",
    len(dashboard_data)
)

print(
    "Total curves:",
    sum(
        len(curves)
        for curves
        in dashboard_data.values()
    )
)

Processing: +ve_cell1part1 (51 curves)
Processing: +ve_cell1part2 (40 curves)
Processing: +ve_cell1part3 (14 curves)
Processing: +ve_cell2part1 (28 curves)
Processing: +ve_cell2part2 (42 curves)
Processing: +ve_cell2part3 (31 curves)
Processing: +ve_cell2part4 (40 curves)
Processing: -ve_cell1part1 (20 curves)
Processing: -ve_cell1part2 (37 curves)
Processing: -ve_cell1part3 (37 curves)
Processing: -ve_cell1part4 (10 curves)
Processing: -ve_cell2part1 (80 curves)
Processing: -ve_cell2part2 (37 curves)
Processing: -ve_cell3part1 (82 curves)
Processing: -ve_cell3part2 (29 curves)
Processing: 20_cell1part1 (93 curves)
Processing: 20_cell1part2 (83 curves)
Processing: 20_cell2part1 (61 curves)
Processing: 20_cell2part2 (83 curves)
Processing: 20_cell3part1 (36 curves)
Processing: 20_cell3part2 (47 curves)
Processing: 20_cell3part3 (25 curves)
Processing: 320_cell1part1 (67 curves)
Processing: 320_cell1part2 (47 curves)
Processing: 320_cell2part1 (58 curves)
Could not read: afm_values/320.x

In [6]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

p = Path("AFM_dashboard.html")

modified_uk = datetime.fromtimestamp(
    p.stat().st_mtime,
    tz=ZoneInfo("Europe/London")
)

print("Last modified UK time:", modified_uk)

Last modified UK time: 2026-08-26 13:02:04.713273+01:00
